# DSCI 521


## Lecture 3 - Installing stuff & the Unix environment


### 2016-09-12

# Learning Objectives

## By the end of the lecture, students should be able to:

- Use environment variables and paths appropriately
- Find out where something is installed
- Set up and personalize the Unix environment

# Environment variables

- environment variables define the unix environment
- some are set by the user, some are set by the system
- by default variables are strings

## Setting your own local shell variables

- variables are set without using `$` sign 
- white space matters
- need the `$` as a prefix when we want to retrieve the variable's stored value
- variables created this way will exist until we exit the shell

In [ ]:
MYVARIABLE="A string I need in memory"
echo $MYVARIABLE

## Storing numbers in shell variables

As mentioned previously, default type for shell variables is a string:

In [ ]:
sum=14+25
echo $sum

You can use the flags to declare a shell variable to be a different type. Here we use the `-i` flag to declare the type as an integer:

In [ ]:
declare -i sum
sum=14+25
echo $sum

For more about performing math in the shell see: http://faculty.salina.k-state.edu/tim/unix_sg/bash/math.html#math

## Finding out what environment variables already exist

When you launch the shell, it reads several files, which among other things, set the value of many environment variables. 

To see what environment variables are currently stored in memory you can use the `env` commmand: 

In [ ]:
env

## Why did I not see the variables I created when I called `env`?
- the variables we created are local shell variables because we did not export them
- to make variables available to other programs and child/sub shells we can use the export command when we define them

In [ ]:
export MYGLOBALVAR="Another string I need in memory"
echo $MYGLOBALVAR

In [ ]:
env

## Where do these other environment variables come from?

When you open a new bash shell, it looks for and executes commands from a series of files, if they exist. Examples of these include:

1. `/etc/profile`
2. `~/.bash_profile`
3. `~/.bashrc`
3. `~/.bash_login`
4. `~/.profile`

It is the commands within these files which specify and define the shell environment variables, including `PATH`, `HOME`, `PS1`, etc.  

## Where do these other environment variables come from?

Of these files, we will discuss only `~/.bash_profile` (Mac & Windows) and `~/.bashrc` (Linux) as this is a file that is useful for us to permanently store environment variables we specify, as well as to modify other environment variables (*e.g.,* `PATH`) when we install new programs.

## Exercise 1 - Important Shell variables

From the list of Shell environment variables below, use `echo $VARIABLE_NAME` to find out the value of the variable on your computer, and if necessary, use Google to findout what the purpose of the variable is:

1. PATH
2. HOME
3. PWD
4. PS1
5. OLDPWD

# Paths

To run any command from the shell, the shell needs to know where the command is stored/saved before it can find it to run it.

The `PATH` is a special environment variable which specifies locations where the shell should look for commands.

To see the value of your path, you can type:

In [ ]:
echo $PATH

Each path, separated by a colon, is a place on the computer where the shell can look for commands. 

As long as a command/script/program lives in one of these places, then you can simply state the name of the command/script/program instead of typing the path to the program + program name.

## Simple example - adding a directory containining a routinely used script to your `PATH`

- Let's download a "useful" unix shell script, "install it", and add it to our PATH. 
- In doing so, we can use it anywhere on our computer without directly pointing the the scripts location.
- Saves us some typing the long run.
- This script is called nice_number and was created by Dave Taylor and can be found in the book ["Wicked Cool Shell Scripts"](http://intuitive.com/wicked/index.shtml)

Let's first make directory for the script, and then download and save it there:

In [ ]:
mkdir scripts 

In [ ]:
cd scripts
pwd

In [ ]:
curl -o nice_number http://intuitive.com/wicked/scripts/004-nicenumber.txt

To run the script we need to call bash and if we are not in the scripts directory, we need to point to it:

In [ ]:
bash nice_number

In [ ]:
cd ..
bash nice_number

In [ ]:
bash scripts/nice_number

### Some script-related housekeeping to make it executable

- ensure the script has a shebang
- change the permissions using `chmod +x`

In [ ]:
cd scripts
head nice_number

In [ ]:
chmod +x nice_number

Now we can call the script without calling `bash`, but we still need to point to it...

In [ ]:
./nice_number

### Back to dealing with the `PATH`

- We can fix the problem of needing to point to the script by adding the directory it lives in to the path
- We do this by permanently modifying the `PATH` variable in the `.bash_profile` file as so:

```
export PATH="$PATH:/absolute_path_to_nice_number"
```

**note - after we do this, we must restart the ternimal/kernel/shell before this will take effect!**

In [ ]:
pwd

In [ ]:
# if you are using Mac or Windows
atom ~/.bash_profile

# if you are using Linux
atom ~/.bashrc

### Calling the script without specifying the path to the script

After adding the absolute path to the PATH environment variable in `~/.bash_profile` we should now be able to call the script without explicitly pointing to it:

In [ ]:
nice_number

## A few best practices about adding things to the `PATH`

- If you are adding only a single executable file (*e.g.,* script), consider putting that file in a directory that is already in your PATH (*e.g.,* `$HOME/bin`) to avoid creating a horrendous PATH definition.
- If there are multiple executable files that need to be added to the `PATH`, and they need to be together in a single directory, it is then advisable to add this directory to your `PATH`.
- When you add a directory to your `PATH`, it is advisable to use `export PATH="$PATH:/the_absolute_path"` as opposed to `export PATH="/the_absolute_path":`PATH="$PATH:the_absolute_path"` so that your additions do not come before original system files.

## symlink - an alternative strategy to manage the `PATH`

**symlink** - a file which contains a text string that is automatically interpreted and followed by the operating system as a path to another file or directory (target). 

- if a symlink is deleted, its target remains unaffected
- if target is moved, renamed or deleted, the symbolic link is not automatically updated or deleted, but continues to exist and still points to the old target
- created with the `ln` command:

```
ln -s {/path/to/file-name} {link-name}
```

- ideally the path used in the link name is already in your PATH variable 

*Curious about the difference between a symlink/softlink and a hardlink? See [this blog post](http://www.maxi-pedia.com/difference+between+symbolic+link+and+hard+link) for a nice and simple explanation.*

### Making `nice_number` work using a symlink

Lets remove the `nice_number` script from the `PATH` and then add a symlink in its stead. *Remember to quit & restart the shell for changes to take effect*

In [ ]:
# if you are using Mac or Windows
atom ~/.bash_profile

# if you are using Linux
atom ~/.bashrc

After removing `nice_number` from the `PATH` by deleting it from the `.bash_profile` or `.bashrc`, we can no longer call `nice_number` from the command line without directly pointing to it:

In [ ]:
nice_number

In [ ]:
./scripts/nice_number

Now we use the `ln` command to add a symlink so that we can again call `nice_number` without explicitly pointing to it:

In [ ]:
ln -s $PWD/scripts/nice_number /usr/local/bin

*note - if the above doesn't work, try the same command using `sudo` proceeding it:*

`sudo ln -s $PWD/scripts/nice_number /usr/local/bin`


We can see that we created a file called `nice_number` in `/usr/local/bin`, and using the `readlink` command, we can see that it contains the absolute path to the `nice_number` script:

In [ ]:
readlink /usr/local/bin/nice_number

Unlike adding the absolute path to `nice_number` to the `PATH` variable via `~/.bash_profile` or `~/.bashrc`, we do not need to restart the shell for this to take effect:

In [ ]:
nice_number

###### Note for Windows users:

*Some Windows users were not able to follow the intructions laid out above and have it work successfully, below is a fix that helped some solve this probem:*

1. Open gitbash with admin privileges (search for Gitbash, right click it and Run as Admin) 
2. add `/c/` to the scripts path like this: 
```
ls -s /c/Users/rorui/Documents/scripts/nice_number2 /usr/local/bin
```

## Exercise 2 - accessing Sublime text from the command line

- Pair up into groups of 4
- Download the Sublime text editor from https://www.sublimetext.com/
- Two people from each group will partner up to make Sublime to be able to be called from the command line using the command subl by adding it to the PATH, while the other two people will use a symlink to the same purpose
- Share your working solution with your group

# Finding out where something is installed

- Sometimes installers are used to install programs (e.g., Homebrew, Anaconda's Python distribution, CRAN's R)
- If you need to do some troubleshooting, you need to ask where these things were installed
- The `which` command is your friend for this task:

In [ ]:
which Atom

## Exercise 3 - Program scavenger hunt

#### Part A

Using the command `which`, discover where the following programs are installed on your computer:

1. Python
2. R
3. Git
4. Bash
5. Atom

#### Part B

Does `which` actually tell you where something is installed? What about programs that use symlinks to tell the shell where to look for them? Do any of the programs use symlinks? *Hint - use the `readlink` command.*

# Setting up and personalizing the Unix environment

- There are several modifications you can make to your Bash Shell to personalize it to your taste
- Examples include, modifying the primary and secondary prompts and colour coding text
- We will walkthrough a couple of useful personalizations

## Customizing the primary prompt

- `PS1` is the environment variable that controls your prompt
- The standard value for `PS1` is usually something like: `\h:\W \u\$` which results in the output of: 

`{hostname}:{current_directory} {username}$` 

which on my computer is something like: `Tiffanys-MacBook-Pro:Documents tiffanytimbers$`

We can change the prompt to a simple `$` by reassigning `PS1` by:

```
PS1="$ "
```

To make this change permanent, we would put this assignemt in the `~/.bash_profile` file (don't forget to lead with `export`!). 

*Remember to quit & restart the shell for changes to take effect*

## Exercise 4 - Customize your prompt

1. Visit http://ss64.com/bash/syntax-prompt.html to view many of the possible options to customize your prompt.
2. Modify `PS1` in `~/.bash_profile` to make a permanent change.
3. Share your new prompt and how you created it with the person beside you.

## Customize the colors of files and folders

- `LSCOLORS` (Mac & Windows) and `LS_COLORS` (Linux) is the environment variable that controls the color of the text output from `ls`
- There may be no default values for `LSCOLORS`/`LS_COLORS`, but you can set them the same way you set new values for `PS1`.
- To enable this option on Mac OSX, you also need to set `CLICOLOR=1` in the `~/.bash_profile`.

Resources for what values to set `LSCOLORS`/`LS_COLORS` to:

- `LSCOLORS` (Mac & Windows) - http://osxdaily.com/2012/02/21/add-color-to-the-terminal-in-mac-os-x/
- `LS_COLORS` (Linux) - http://linux-sxs.org/housekeeping/lscolors.html

## Exercise 5 - Customize the colors of files and folders

1. Modify `LSCOLORS`/`LS_COLORS` in `~/.bash_profile` to make a permanent change.
2. Share your new prompt and how you created it with the person beside you.

# Concept map wrap-up
- environment variables and paths
- where something is installed
- personalizing the Unix environment